# 06 Notebook Workflows and Modular Code

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Reuse code across notebooks the right way: Python <b>modules</b> (<code>.py</code> files imported with <code>import</code>), <code>%run</code> to include a notebook, and <code>dbutils.notebook.run</code> to call a notebook as a separate unit with parameters, a timeout and a return value, including running several in parallel. Know when to use each, and when to use job tasks instead.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Copy-pasted helper code across 20 notebooks is how pipelines become unmaintainable. Databricks gives you three ways to share and chain code, with very different behavior: shared variables or not, parameters or not, parallelism or not. Interviewers often ask "<code>%run</code> vs <code>dbutils.notebook.run</code>?", and the exam expects you to know when orchestration belongs in a job instead.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Three ways to reuse code</b><br>
| | <code>import module</code> (<code>.py</code> file) | <code>%run ./other_notebook</code> | <code>dbutils.notebook.run("./other", 600, {...})</code> |<br>|---|---|---|---|<br>| What it does | Imports functions and classes from a Python file | Runs another notebook <b>inline</b>, in the current context | Runs another notebook as a <b>separate run</b> |<br>| Shares variables | Only what you import | Yes, everything it defines | No. Communicates via parameters (widgets) and <code>dbutils.notebook.exit</code> |<br>| Parameters | Function arguments | No | Yes (a dict of strings) |<br>| Return value | Function return values | n/a | One string from <code>dbutils.notebook.exit</code> |<br>| Parallel | n/a | No | Yes, with threads |<br>| Testable with pytest | <b>Yes</b> | No | No |<br>| Best for | Shared logic (recommended) | Config and setup notebooks | Simple notebook chaining inside a notebook |<br><br>
For orchestration of real pipelines, prefer <b>job tasks</b> (lesson 02): they give retries, repair, monitoring and parallelism per task.
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A team's 30 notebooks each started with <code>%run ../utils/helpers</code>, a 1,500-line notebook. Every <code>%run</code> silently overwrote variables like <code>df</code> and <code>path</code> in the caller, causing bugs that were hard to trace, and none of the helpers could be unit-tested. They moved the helpers into a package (<code>src/pipeline_utils/*.py</code>), imported only what each notebook needs, added pytest tests, and kept <code>%run</code> only for one tiny <code>config</code> notebook.
</div>

## Syntax

```python
# Module next to the notebook (Git folder or workspace files): src/order_utils.py
from src.order_utils import clean_orders

%run ./config                                    # alone in its cell; shares variables

result = dbutils.notebook.run("./child", 600, {"run_date": "2024-11-01"})   # timeout in seconds
# in the child:  dbutils.widgets.get("run_date") ... dbutils.notebook.exit(json.dumps({...}))
```

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates the schema and a folder for the module and the child notebook: a workspace folder on Databricks, a temporary folder locally.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The folder used for the demo files.
</div>

In [0]:
CATALOG = "workspace"
DB = f"{CATALOG}.workflows"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")
me = spark.sql("SELECT current_user()").first()[0]
CODE_DIR = f"/Workspace/Users/{me}/databricks-learning-workflows/lib"

In [0]:
import os, sys, json, importlib
os.makedirs(CODE_DIR, exist_ok=True)
print(CODE_DIR)

## 1. Shared logic as a Python module

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes a module <code>order_utils.py</code> with two transformation functions, adds its folder to <code>sys.path</code>, imports it and uses it.<br><br>
<b>Why it matters</b><br>Modules are plain Python: you import exactly what you need, nothing leaks into your notebook, and the same functions can be tested with pytest (lesson 07) and reused by jobs. In a Git folder, the notebook's own directory and the repo root are already importable, so <code>from src.order_utils import ...</code> works without touching <code>sys.path</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
3 cleaned orders with a <code>size</code> column.
</div>

In [0]:
MODULE = '''
from pyspark.sql import DataFrame, functions as F

def clean_orders(df: DataFrame) -> DataFrame:
    """Drop rows without an id or with a non-positive amount, and normalize the country code."""
    return (df.filter(F.col("order_id").isNotNull() & (F.col("amount") > 0))
              .withColumn("country", F.upper(F.trim("country"))))

def add_size(df: DataFrame, threshold: float = 100.0) -> DataFrame:
    return df.withColumn("size", F.when(F.col("amount") >= threshold, "large").otherwise("small"))
'''
with open(f"{CODE_DIR}/order_utils.py", "w") as f:
    f.write(MODULE)

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
import order_utils
importlib.reload(order_utils)                     # pick up edits without restarting Python

raw = spark.createDataFrame([(1, " in ", 120.0), (2, "uk", 30.0), (None, "ae", 50.0), (4, "ae", -1.0), (5, "Ae", 80.0)],
                            "order_id INT, country STRING, amount DOUBLE")
order_utils.add_size(order_utils.clean_orders(raw)).orderBy("order_id").show()

## 2. `%run`: include a notebook

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Includes the course's <code>00_helper</code> notebook with <code>%run</code>, which defines a <code>greet()</code> function in this notebook's context, and calls it.<br><br>
<b>Why it matters</b><br><code>%run</code> is like pasting the other notebook's code here. It's convenient for a small config or setup notebook, but everything it defines lands in your namespace, and it takes no parameters. It must be alone in its cell, with a literal path.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
This works when the repository is cloned as a Git folder (or imported with its folder structure), because the path is relative to this notebook.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
No output from the <code>%run</code> cell, then <code>Hello, workflows!</code>.
</div>

In [0]:
%run ../00_platform_tour/00_helper

In [0]:
print(greet("workflows"))

## 3. `dbutils.notebook.run`: call a notebook with parameters

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a child notebook that reads a <code>country</code> and a <code>min_amount</code> parameter, computes a small result, and returns it as JSON with <code>dbutils.notebook.exit</code>. Then calls it from here.<br><br>
<b>Why it matters</b><br>The child runs separately: it can't see this notebook's variables, and this notebook only gets back the string it returns. Pass small values as parameters and return small values. Larger results go through tables.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
<code>dbutils.notebook.run</code> works on serverless notebooks. If it isn't available in your workspace, the cell prints a message.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A dictionary like <code>{"country": "IN", "orders": ..., "revenue": ...}</code> parsed from the child's JSON.
</div>

In [0]:
import base64
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import workspace

w = WorkspaceClient()
FOLDER = f"/Users/{me}/databricks-learning-workflows"
SALES = f"{DB}.nb_sales"
spark.sql(f"CREATE OR REPLACE TABLE {SALES} AS SELECT * FROM VALUES ('IN', 120.0), ('IN', 40.0), ('UK', 75.0), ('AE', 300.0), ('UK', 15.0) AS t(country, amount)")

CHILD = f'''# Databricks notebook source
import json
dbutils.widgets.text("country", "IN")
dbutils.widgets.text("min_amount", "0")
country, min_amount = dbutils.widgets.get("country"), float(dbutils.widgets.get("min_amount"))
rows = spark.table("{SALES}").filter(f"country = '{{country}}' AND amount >= {{min_amount}}")
result = {{"country": country, "orders": rows.count(), "revenue": rows.agg({{"amount": "sum"}}).first()[0]}}
dbutils.notebook.exit(json.dumps(result))
'''
child_path = f"{FOLDER}/nb_child_country_summary"
w.workspace.mkdirs(FOLDER)
w.workspace.import_(path=child_path, format=workspace.ImportFormat.SOURCE, language=workspace.Language.PYTHON,
                    content=base64.b64encode(CHILD.encode()).decode(), overwrite=True)

try:
    out = dbutils.notebook.run(child_path, 600, {"country": "IN", "min_amount": "0"})
    print(json.loads(out))
except Exception as e:
    print("dbutils.notebook.run not available here:", type(e).__name__, str(e)[:200])

## 4. Run several notebooks in parallel

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Calls the child notebook for three countries at the same time with a thread pool, and collects the results.<br><br>
<b>Why it matters</b><br><code>dbutils.notebook.run</code> blocks until the child finishes, so a loop runs children one after another. Threads run them concurrently. For production fan-out, a job's <b>for each</b> task does the same with monitoring and repair.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
One result per country, in about the time of a single run rather than three.
</div>

In [0]:
import time
from concurrent.futures import ThreadPoolExecutor

def summarize(country):
    return json.loads(dbutils.notebook.run(child_path, 600, {"country": country, "min_amount": "20"}))

try:
    start = time.time()
    with ThreadPoolExecutor(max_workers=3) as pool:
        results = list(pool.map(summarize, ["IN", "UK", "AE"]))
    for r in results:
        print(r)
    print(f"elapsed: {time.time() - start:.0f}s")
except Exception as e:
    print("Parallel notebook runs not available here:", type(e).__name__, str(e)[:200])

## 5. Variables leak with `%run`, not with modules

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Simulates what <code>%run</code> does (executing another notebook's code in this namespace) and shows that it overwrites a variable you already had. Then shows that a module import doesn't.<br><br>
<b>Why it matters</b><br>This is the most common <code>%run</code> bug: a helper notebook defines <code>df</code> or <code>path</code>, and your notebook's variable silently changes.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
After the simulated <code>%run</code>, <code>path</code> has changed. After the import, it's unchanged.
</div>

In [0]:
path = "/Volumes/workspace/workflows/files/my_input"
helper_notebook_code = 'path = "/tmp/helper_scratch"\ndef helper(): return 42'
exec(helper_notebook_code, globals())            # what %run effectively does
print("after %run-style include:", path)

path = "/Volumes/workspace/workflows/files/my_input"
with open(f"{CODE_DIR}/helper_module.py", "w") as f:
    f.write(helper_notebook_code)
import helper_module
importlib.reload(helper_module)
print("after import:            ", path, "| helper() =", helper_module.helper())

## Under the hood

```
import order_utils      ─▶ Python loads the .py file once (cached in sys.modules; reload to pick up edits)
%run ./x                ─▶ cells of x executed in THIS notebook's Python context (same variables, same Spark session)
dbutils.notebook.run    ─▶ new ephemeral notebook run (own context, widgets = arguments, timeout)
                           └─ returns the string passed to dbutils.notebook.exit(...)
job tasks               ─▶ separate task runs with retries, repair, run history (preferred for pipelines)
```

In [0]:
print(sorted(f for f in os.listdir(CODE_DIR) if f.endswith(".py")))

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: <code>%run</code> with other code in the same cell</b><br>
<code>%run</code> must be alone in its cell.<br><br>
**⛔ Problem: <code>%run</code> with a variable path**<br>The path must be literal. Use <code>dbutils.notebook.run</code> (or imports) for dynamic choices.<br><br>
<b>⛔ Problem: a module change isn't picked up</b><br>Python caches imports. Use <code>importlib.reload(module)</code> or restart Python (<code>dbutils.library.restartPython()</code>).<br><br>
**⛔ Problem: returning a DataFrame from <code>dbutils.notebook.run</code>**<br>Only a string comes back. Write data to a table and return its name, or return small JSON.<br><br>
**⛔ Problem: long chains of <code>dbutils.notebook.run</code> as a pipeline**<br>No retries per step, no repair, hard to monitor. Use a multi-task job.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is the difference between <code>%run</code> and <code>dbutils.notebook.run</code>?</b><br>
<code>%run</code> executes another notebook inline in the current context, so its variables and functions become available, but it takes no parameters and returns nothing. <code>dbutils.notebook.run</code> starts the notebook as a separate run with parameters (as widgets) and a timeout, doesn't share variables, and returns the string passed to <code>dbutils.notebook.exit</code>. It can also be run in parallel from threads.<br><br>

<b>🎤 2. How do you share code across notebooks in a maintainable way?</b><br>
Put the logic in Python modules or packages (files in a Git folder, or a wheel), import what's needed, and unit-test it with pytest. <code>%run</code> is fine for a small config notebook, and jobs handle orchestration.<br><br>

<b>🎤 3. How can a child notebook return results?</b><br>
With <code>dbutils.notebook.exit(value)</code>, which returns one string, often JSON. Larger results go to a table, and the child returns the table name or a status.<br><br>

<b>🎤 4. When would you use <code>dbutils.notebook.run</code> instead of a job with several tasks?</b><br>
For simple, dynamic chaining inside an interactive or one-task workflow, for example running the same notebook for a list computed at run time. For production pipelines, a multi-task job (with for each tasks for fan-out) is better because of retries, repair and monitoring.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A notebook must run another notebook with a <code>date</code> parameter and use the value it returns. Which approach works?</b><br>
A. <code>%run ./child date=2024-11-01</code><br>
B. <code>dbutils.notebook.run("./child", 600, {"date": "2024-11-01"})</code>, with the child calling <code>dbutils.notebook.exit(...)</code><br>
C. <code>spark.sql("RUN NOTEBOOK child")</code><br>
D. <code>import child</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. After <code>%run ./setup</code>, a variable <code>config</code> defined in <code>setup</code> is available in the calling notebook. Why?</b><br>
A. <code>%run</code> executes the notebook in the caller's context<br>
B. Variables are global across the workspace<br>
C. <code>setup</code> returned it with <code>dbutils.notebook.exit</code><br>
D. Unity Catalog shares it<br>
<details><summary><b>Show answer</b></summary><b>A.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Add a function <code>revenue_by_country(df)</code> to <code>order_utils.py</code>, reload the module and use it on the cleaned orders</li><li>Write a helper <code>run_all(fn, items, workers)</code> that applies any function to a list in parallel with a thread pool, and test it with a simple function locally</li><li>(Databricks) Use <code>run_all</code> with <code>summarize</code> for <code>["IN", "UK", "AE", "US"]</code>, and handle a country with no orders</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Extend the module and reload
with open(f"{CODE_DIR}/order_utils.py", "a") as f:
    f.write('''
def revenue_by_country(df: DataFrame) -> DataFrame:
    return df.groupBy("country").agg(F.round(F.sum("amount"), 2).alias("revenue"))
''')
importlib.reload(order_utils)
order_utils.revenue_by_country(order_utils.clean_orders(raw)).orderBy("country").show()

# 2. Generic parallel helper
from concurrent.futures import ThreadPoolExecutor
def run_all(fn, items, workers=4):
    with ThreadPoolExecutor(max_workers=workers) as pool:
        return dict(zip(items, pool.map(fn, items)))
print(run_all(lambda x: x * x, [1, 2, 3]))

In [0]:
# 3. Parallel notebook runs; US has no orders, so revenue is None
try:
    for country, r in run_all(summarize, ["IN", "UK", "AE", "US"], workers=4).items():
        print(country, "->", r if r["orders"] else "no orders")
except Exception as e:
    print("Not available here:", type(e).__name__)

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Shared logic belongs in Python modules: import, reload, test</li><li><code>%run</code>: inline include, shares variables, no parameters. Keep it for small config notebooks</li><li><code>dbutils.notebook.run</code>: separate run, parameters as widgets, timeout, one string returned via <code>exit</code></li><li>Run children in parallel with threads. In production, use job tasks and for each instead</li></ul>
</div>

| Need | Use |
|---|---|
| Reuse functions | `from module import fn` |
| Pick up module edits | `importlib.reload(module)` |
| Include setup code | `%run ./config` |
| Call with parameters | `dbutils.notebook.run(path, timeout, {...})` |
| Return a value | `dbutils.notebook.exit(json.dumps(...))` |
| Fan out | `ThreadPoolExecutor` or a for each task |

## Git commit

```
git add 07_workflows/06_notebook_workflows.ipynb
git commit -m "add 07_06 notebook workflows notebook"
```